# BizInsight - Intelligent Recommendation Engine

This notebook implements a recommendation engine
that transforms business analysis results into
actionable business recommendations.

The engine uses financial indicators, historical
sales patterns, profitability analysis and predefined
business rules to identify opportunities, risks
and potential improvements.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

project_dir = Path.cwd()

if project_dir.name == "notebooks":
    project_dir = project_dir.parent

analysis_dir = project_dir / "results" / "business_analysis"

analysis_data = {}

for file in sorted(analysis_dir.glob("*.csv")):
    analysis_data[file.stem] = pd.read_csv(file)
    print(f"Loaded: {file.name}")

print("\nRecommendation engine initialized successfully.")
print("Analysis datasets loaded:", len(analysis_data))

Loaded: business_kpis.csv
Loaded: daily_performance.csv
Loaded: hidden_high_margin.csv
Loaded: high_return_products.csv
Loaded: loss_making_products.csv
Loaded: monthly_performance.csv
Loaded: popular_low_margin.csv
Loaded: product_performance.csv
Loaded: promotion_performance.csv
Loaded: return_analysis.csv
Loaded: sales_anomalies.csv
Loaded: store_performance.csv

Recommendation engine initialized successfully.
Analysis datasets loaded: 12


## Product Profitability Recommendations

Generate actionable recommendations based on product
sales volume, gross profit and profit margins.

Each recommendation contains a detected finding,
supporting evidence, a suggested action and a priority.

In [2]:
def generate_product_recommendations(data):

    recommendations = []

    for _, row in data["popular_low_margin"].iterrows():

        recommendations.append({
            "Category": "Product Profitability",
            "Type": "Popular but Low Margin",
            "Entity": row["ProductID"],
            "Priority": "High",
            "Finding": (
                f"Product sold {row['NetUnitsSold']:.0f} net units "
                f"with a gross margin of "
                f"{row['GrossMargin']:.2f}%."
            ),
            "Recommendation": (
                "Review the selling price and supplier costs. "
                "Consider testing a small price increase "
                "while monitoring sales volume."
            )
        })

    for _, row in data["hidden_high_margin"].iterrows():

        recommendations.append({
            "Category": "Product Profitability",
            "Type": "Hidden High Margin",
            "Entity": row["ProductID"],
            "Priority": "Medium",
            "Finding": (
                f"Product has a gross margin of "
                f"{row['GrossMargin']:.2f}% but sold only "
                f"{row['NetUnitsSold']:.0f} net units."
            ),
            "Recommendation": (
                "Improve product visibility and test targeted "
                "promotions while protecting profitability."
            )
        })

    for _, row in data["loss_making_products"].iterrows():

        recommendations.append({
            "Category": "Product Profitability",
            "Type": "Loss-Making Product",
            "Entity": row["ProductID"],
            "Priority": "High",
            "Finding": (
                f"Product generated a gross loss of "
                f"{abs(row['GrossProfit']):,.2f}."
            ),
            "Recommendation": (
                "Investigate pricing, discounts, returns "
                "and supplier costs before deciding "
                "whether to discontinue the product."
            )
        })

    return pd.DataFrame(recommendations)


product_recommendations = generate_product_recommendations(
    analysis_data
)

print("Product recommendations generated:",
      len(product_recommendations))

display(product_recommendations.head(10))

Product recommendations generated: 339


,Category,Type,Entity,Priority,Finding,Recommendation
0,Product Profitability,Popular but Low Margin,PROD-100358,High,Product sold 263 net units with a gross margin...,Review the selling price and supplier costs. C...
1,Product Profitability,Popular but Low Margin,PROD-100357,High,Product sold 213 net units with a gross margin...,Review the selling price and supplier costs. C...
2,Product Profitability,Popular but Low Margin,PROD-100356,High,Product sold 188 net units with a gross margin...,Review the selling price and supplier costs. C...
3,Product Profitability,Popular but Low Margin,PROD-141937,High,Product sold 138 net units with a gross margin...,Review the selling price and supplier costs. C...
4,Product Profitability,Popular but Low Margin,PROD-159723,High,Product sold 137 net units with a gross margin...,Review the selling price and supplier costs. C...
5,Product Profitability,Popular but Low Margin,PROD-100359,High,Product sold 132 net units with a gross margin...,Review the selling price and supplier costs. C...
6,Product Profitability,Popular but Low Margin,PROD-147352,High,Product sold 128 net units with a gross margin...,Review the selling price and supplier costs. C...
7,Product Profitability,Popular but Low Margin,PROD-141938,High,Product sold 115 net units with a gross margin...,Review the selling price and supplier costs. C...
8,Product Profitability,Popular but Low Margin,PROD-147351,High,Product sold 114 net units with a gross margin...,Review the selling price and supplier costs. C...
9,Product Profitability,Popular but Low Margin,PROD-100355,High,Product sold 110 net units with a gross margin...,Review the selling price and supplier costs. C...


## Promotion and Discount Recommendations

Analyze the profitability of promotions, markdowns
and clearance sales to identify unprofitable discount
strategies and recommend improvements.

In [3]:
def generate_promotion_recommendations(data):

    recommendations = []

    for _, row in data["promotion_performance"].iterrows():

        if row["SalesType"] == "Full Price":
            continue

        if row["GrossProfit"] < 0:

            priority = "High"
            recommendation = (
                "Review the discount strategy and product costs. "
                "Consider reducing discount depth or limiting "
                "discounts to selected products."
            )

        elif row["GrossMargin"] < 15:

            priority = "Medium"
            recommendation = (
                "Review discount levels and test alternatives "
                "that maintain a healthier profit margin."
            )

        else:
            continue

        recommendations.append({
            "Category": "Promotion Performance",
            "Type": "Low Promotion Profitability",
            "Entity": row["SalesType"],
            "Priority": priority,
            "Finding": (
                f"Generated net sales of {row['NetSales']:,.2f} "
                f"and gross profit of {row['GrossProfit']:,.2f}, "
                f"with a gross margin of "
                f"{row['GrossMargin']:.2f}%."
            ),
            "Recommendation": recommendation
        })

    return pd.DataFrame(recommendations)


promotion_recommendations = (
    generate_promotion_recommendations(analysis_data)
)

print("Promotion recommendations generated:",
      len(promotion_recommendations))

display(promotion_recommendations)

Promotion recommendations generated: 3


,Category,Type,Entity,Priority,Finding,Recommendation
0,Promotion Performance,Low Promotion Profitability,Markdown Tier 1,Medium,"Generated net sales of 177,712.44 and gross pr...",Review discount levels and test alternatives t...
1,Promotion Performance,Low Promotion Profitability,Markdown Tier 2,High,"Generated net sales of 77,602.53 and gross pro...",Review the discount strategy and product costs...
2,Promotion Performance,Low Promotion Profitability,Clearance,High,"Generated net sales of 17,408.76 and gross pro...",Review the discount strategy and product costs...


## Product Return Recommendations

Identify products with high return rates and generate
actionable recommendations to reduce returns and
improve business profitability.

In [4]:
def generate_return_recommendations(data):

    recommendations = []

    for _, row in data["high_return_products"].iterrows():

        priority = (
            "High" if row["ReturnRate"] >= 20
            else "Medium"
        )

        recommendations.append({
            "Category": "Product Returns",
            "Type": "High Return Rate",
            "Entity": row["ProductID"],
            "Priority": priority,
            "Finding": (
                f"Product recorded {row['UnitsReturned']:.0f} "
                f"returned units out of "
                f"{row['UnitsSold']:.0f} units sold, "
                f"with a return rate of "
                f"{row['ReturnRate']:.2f}% and a total "
                f"return value of {row['ReturnValue']:,.2f}."
            ),
            "Recommendation": (
                "Investigate the reasons for product returns. "
                "Review product quality, sizing accuracy, "
                "product descriptions and customer feedback. "
                "Consider discussing recurring issues "
                "with the supplier."
            )
        })

    return pd.DataFrame(recommendations)


return_recommendations = (
    generate_return_recommendations(analysis_data)
)

print("Return recommendations generated:",
      len(return_recommendations))

display(return_recommendations.head(10))

Return recommendations generated: 347


,Category,Type,Entity,Priority,Finding,Recommendation
0,Product Returns,High Return Rate,PROD-168539,Medium,Product recorded 26 returned units out of 172 ...,Investigate the reasons for product returns. R...
1,Product Returns,High Return Rate,PROD-168538,High,Product recorded 25 returned units out of 107 ...,Investigate the reasons for product returns. R...
2,Product Returns,High Return Rate,PROD-140876,Medium,Product recorded 18 returned units out of 171 ...,Investigate the reasons for product returns. R...
3,Product Returns,High Return Rate,PROD-168540,Medium,Product recorded 17 returned units out of 155 ...,Investigate the reasons for product returns. R...
4,Product Returns,High Return Rate,PROD-168533,Medium,Product recorded 17 returned units out of 169 ...,Investigate the reasons for product returns. R...
5,Product Returns,High Return Rate,PROD-184560,Medium,Product recorded 9 returned units out of 72 un...,Investigate the reasons for product returns. R...
6,Product Returns,High Return Rate,PROD-184558,Medium,Product recorded 9 returned units out of 83 un...,Investigate the reasons for product returns. R...
7,Product Returns,High Return Rate,PROD-100356,Medium,Product recorded 24 returned units out of 212 ...,Investigate the reasons for product returns. R...
8,Product Returns,High Return Rate,PROD-167643,Medium,Product recorded 17 returned units out of 101 ...,Investigate the reasons for product returns. R...
9,Product Returns,High Return Rate,PROD-181883,Medium,Product recorded 13 returned units out of 80 u...,Investigate the reasons for product returns. R...


## Business Performance Recommendations

Generate actionable recommendations based on declining
monthly sales, underperforming stores and unusual
daily sales patterns.

In [5]:
def generate_business_recommendations(data):

    recommendations = []

    # Monthly performance
    for _, row in data["monthly_performance"].iterrows():

        if pd.isna(row["SalesGrowth"]):
            continue

        if row["SalesGrowth"] <= -10:

            recommendations.append({
                "Category": "Business Performance",
                "Type": "Declining Monthly Sales",
                "Entity": str(row["Month"]),
                "Priority": "High",
                "Finding": (
                    f"Monthly net sales decreased by "
                    f"{abs(row['SalesGrowth']):.2f}% "
                    f"compared to the previous month."
                ),
                "Recommendation": (
                    "Investigate changes in customer demand, "
                    "product availability and promotional activity. "
                    "Consider testing targeted sales campaigns."
                )
            })

    # Store performance
    stores = data["store_performance"]

    average_daily_profit = stores["AverageDailyProfit"].median()

    for _, row in stores.iterrows():

        if row["AverageDailyProfit"] < average_daily_profit * 0.70:

            recommendations.append({
                "Category": "Store Performance",
                "Type": "Underperforming Store",
                "Entity": row["Store"],
                "Priority": "Medium",
                "Finding": (
                    f"Average daily gross profit is "
                    f"{row['AverageDailyProfit']:,.2f}, "
                    f"compared to the store median of "
                    f"{average_daily_profit:,.2f}."
                ),
                "Recommendation": (
                    "Review store operations, product assortment "
                    "and local customer demand. Compare performance "
                    "with higher-performing stores."
                )
            })

    # Sales anomalies
    for _, row in data["sales_anomalies"].iterrows():

        deviation = row["SalesDeviation"]

        if deviation <= -50:

            recommendation = (
                "Investigate potential stock shortages, "
                "store closures, reporting issues or "
                "changes in customer demand."
            )

        elif deviation >= 100:

            recommendation = (
                "Investigate the factors behind unusually "
                "high sales. Consider whether successful "
                "promotions or demand patterns can be repeated."
            )

        else:
            continue

        recommendations.append({
            "Category": "Sales Anomalies",
            "Type": row["AnomalyType"],
            "Entity": str(row["Date"]),
            "Priority": "High" if deviation < 0 else "Medium",
            "Finding": (
                f"Daily net sales were {row['NetSales']:,.2f}, "
                f"compared to expected sales of "
                f"{row['ExpectedSales']:,.2f}. "
                f"Sales deviation: {deviation:+.2f}%."
            ),
            "Recommendation": recommendation
        })

    return pd.DataFrame(recommendations)


business_recommendations = (
    generate_business_recommendations(analysis_data)
)

print("Business recommendations generated:",
      len(business_recommendations))

display(business_recommendations.head(15))

Business recommendations generated: 47


,Category,Type,Entity,Priority,Finding,Recommendation
0,Business Performance,Declining Monthly Sales,2025-09,High,Monthly net sales decreased by 43.95% compared...,"Investigate changes in customer demand, produc..."
1,Business Performance,Declining Monthly Sales,2026-03,High,Monthly net sales decreased by 10.08% compared...,"Investigate changes in customer demand, produc..."
2,Store Performance,Underperforming Store,STR-1237,Medium,"Average daily gross profit is 254.90, compared...","Review store operations, product assortment an..."
3,Store Performance,Underperforming Store,STR-1335,Medium,"Average daily gross profit is 243.82, compared...","Review store operations, product assortment an..."
4,Store Performance,Underperforming Store,STR-1353,Medium,"Average daily gross profit is 228.80, compared...","Review store operations, product assortment an..."
5,Store Performance,Underperforming Store,STR-1369,Medium,"Average daily gross profit is 224.37, compared...","Review store operations, product assortment an..."
6,Store Performance,Underperforming Store,STR-1234,Medium,"Average daily gross profit is 218.69, compared...","Review store operations, product assortment an..."
7,Store Performance,Underperforming Store,STR-1344,Medium,"Average daily gross profit is 203.61, compared...","Review store operations, product assortment an..."
8,Store Performance,Underperforming Store,STR-1105,Medium,"Average daily gross profit is 196.69, compared...","Review store operations, product assortment an..."
9,Store Performance,Underperforming Store,STR-1308,Medium,"Average daily gross profit is 179.64, compared...","Review store operations, product assortment an..."


## Final Recommendation Engine Results

Combine recommendations from all business analysis
modules, organize them by priority, summarize the
results and export them for future application
integration.

In [6]:
all_recommendations = pd.concat(
    [
        product_recommendations,
        promotion_recommendations,
        return_recommendations,
        business_recommendations
    ],
    ignore_index=True
)

priority_order = {
    "High": 1,
    "Medium": 2,
    "Low": 3
}

all_recommendations["PriorityRank"] = (
    all_recommendations["Priority"].map(priority_order)
)

all_recommendations = (
    all_recommendations
    .sort_values("PriorityRank", kind="stable")
    .drop(columns=["PriorityRank"])
    .reset_index(drop=True)
)

all_recommendations.insert(
    0,
    "RecommendationID",
    [
        f"REC-{i:04d}"
        for i in range(1, len(all_recommendations) + 1)
    ]
)

print("Total recommendations:", len(all_recommendations))

print("\nRecommendations by category:")
display(all_recommendations["Category"].value_counts())

print("\nRecommendations by priority:")
display(all_recommendations["Priority"].value_counts())

print("\nTop 10 recommendations:")

with pd.option_context("display.max_colwidth", None):
    display(all_recommendations.head(10))

# Save results
recommendation_dir = (
    project_dir / "results" / "recommendations"
)

recommendation_dir.mkdir(
    parents=True,
    exist_ok=True
)

all_recommendations.to_csv(
    recommendation_dir / "recommendations.csv",
    index=False
)

print("\nRecommendations saved successfully.")

Total recommendations: 736

Recommendations by category:


Category
Product Returns          347
Product Profitability    339
Sales Anomalies           37
Store Performance          8
Promotion Performance      3
Business Performance       2
Name: count, dtype: int64


Recommendations by priority:


Priority
Medium    526
High      210
Name: count, dtype: int64


Top 10 recommendations:


,RecommendationID,Category,Type,Entity,Priority,Finding,Recommendation
0,REC-0001,Product Profitability,Popular but Low Margin,PROD-100358,High,Product sold 263 net units with a gross margin of 22.95%.,Review the selling price and supplier costs. Consider testing a small price increase while monitoring sales volume.
1,REC-0002,Product Profitability,Popular but Low Margin,PROD-100357,High,Product sold 213 net units with a gross margin of 22.77%.,Review the selling price and supplier costs. Consider testing a small price increase while monitoring sales volume.
2,REC-0003,Product Profitability,Popular but Low Margin,PROD-100356,High,Product sold 188 net units with a gross margin of 23.31%.,Review the selling price and supplier costs. Consider testing a small price increase while monitoring sales volume.
3,REC-0004,Product Profitability,Popular but Low Margin,PROD-141937,High,Product sold 138 net units with a gross margin of 24.64%.,Review the selling price and supplier costs. Consider testing a small price increase while monitoring sales volume.
4,REC-0005,Product Profitability,Popular but Low Margin,PROD-159723,High,Product sold 137 net units with a gross margin of 26.55%.,Review the selling price and supplier costs. Consider testing a small price increase while monitoring sales volume.
5,REC-0006,Product Profitability,Popular but Low Margin,PROD-100359,High,Product sold 132 net units with a gross margin of 21.84%.,Review the selling price and supplier costs. Consider testing a small price increase while monitoring sales volume.
6,REC-0007,Product Profitability,Popular but Low Margin,PROD-147352,High,Product sold 128 net units with a gross margin of 10.41%.,Review the selling price and supplier costs. Consider testing a small price increase while monitoring sales volume.
7,REC-0008,Product Profitability,Popular but Low Margin,PROD-141938,High,Product sold 115 net units with a gross margin of 26.77%.,Review the selling price and supplier costs. Consider testing a small price increase while monitoring sales volume.
8,REC-0009,Product Profitability,Popular but Low Margin,PROD-147351,High,Product sold 114 net units with a gross margin of 5.26%.,Review the selling price and supplier costs. Consider testing a small price increase while monitoring sales volume.
9,REC-0010,Product Profitability,Popular but Low Margin,PROD-100355,High,Product sold 110 net units with a gross margin of 24.99%.,Review the selling price and supplier costs. Consider testing a small price increase while monitoring sales volume.



Recommendations saved successfully.
